In [1]:
# ============================================================
# Google Trends Fetcher — Google Colab Ready  (v4: time-aligned)
# Timeframe fixed to 2025-04-01 ~ 2025-05-31
# to align with Reddit data from PullPush (approx. May 2025)
# ============================================================

# ── CELL 1: Install dependencies ──────────────────────────────
!pip install pytrends pandas "urllib3<2.0"

# ── CELL 2: Imports ───────────────────────────────────────────
import time
import requests
import pandas as pd
from datetime import datetime, timedelta
from pytrends.request import TrendReq

# ── CELL 3: Configuration (UPDATED) ───────────────────────────
KEYWORDS    = ["Coachella", "Alo Yoga"]
GEO         = "US"
LANGUAGE    = "en-US"
OUTPUT_FILE = "search_data.csv"
MAX_RETRIES = 4
BASE_WAIT   = 15


# ── CELL 4: Fixed timeframe — aligned with Reddit data ────────
#
# Reddit comments (via PullPush) are from approx. May 2025.
# We fix the Google Trends window to the same period so both
# data sources share the same time axis in the final dashboard.
#
# ❌ Old (dynamic, misaligned with Reddit):
#   timeframe = f"{(today - 90days)} {today}"  →  2026-02-03 ~ 2026-05-04
#
# ✅ New (fixed, aligned with Reddit):
#   timeframe = "2025-04-01 2025-05-31"

timeframe = "2025-04-01 2025-05-31"
print(f"Fetching data from {timeframe} …")

# ── CELL 5: Build a browser-spoofed requests Session ──────────
def get_google_nid_cookie() -> str:
    try:
        r = requests.get(
            "https://www.google.com",
            headers={"User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/124.0.0.0 Safari/537.36"
            )},
            timeout=10,
        )
        nid = r.cookies.get("NID", "")
        if nid:
            print(f"  ✓ NID cookie obtained ({len(nid)} chars)")
        else:
            print("  ⚠️  NID cookie not returned — proceeding without it")
        return nid
    except Exception as e:
        print(f"  ⚠️  Could not fetch NID cookie: {e}")
        return ""

print("Obtaining Google session cookie …")
nid_cookie = get_google_nid_cookie()

session = requests.Session()
session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept":          "application/json, text/plain, */*",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer":         "https://trends.google.com/",
})
if nid_cookie:
    session.cookies.set("NID", nid_cookie, domain=".google.com")

# ── CELL 6: Initialise pytrends with the custom session ───────
pytrends = TrendReq(
    hl=LANGUAGE,
    tz=300,
    requests_args={
        "headers": dict(session.headers),
        # Removed "cookies": dict(session.cookies) to avoid 'multiple values for keyword argument' error
    },
)

# Manually add the NID cookie to pytrends' internal session if accessible.
# pytrends.s is often inaccessible in newer versions, so trying _http.cookies.
if nid_cookie:
    if hasattr(pytrends, 's') and hasattr(pytrends.s, 'cookies'):
        pytrends.s.cookies.set("NID", nid_cookie, domain=".google.com")
    elif hasattr(pytrends, '_http') and hasattr(pytrends._http, 'cookies'):
        pytrends._http.cookies.set("NID", nid_cookie, domain=".google.com")
    else:
        print("  ⚠️  Could not set NID cookie on pytrends' internal session. May encounter rate-limiting.")

# ── CELL 7: Fetch data for each keyword separately ────────────
all_rows = []

for keyword in KEYWORDS:
    print(f"\n→ Fetching: '{keyword}' …")

    df_raw = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            pytrends.build_payload(
                kw_list=[keyword],
                cat=0,
                timeframe=timeframe,
                geo=GEO,
                gprop="",
            )
            df_raw = pytrends.interest_over_time()
            break

        except Exception as e:
            msg  = str(e)
            wait = BASE_WAIT * attempt

            if "429" in msg:
                print(f"  ⚠️  Attempt {attempt}/{MAX_RETRIES} — rate-limited (429).")
            else:
                print(f"  ⚠️  Attempt {attempt}/{MAX_RETRIES} failed: {e}")

            if attempt < MAX_RETRIES:
                print(f"     Waiting {wait}s before retry …")
                time.sleep(wait)
            else:
                print(f"  ✗  All retries exhausted for '{keyword}'. Skipping.")

    if df_raw is None or df_raw.empty:
        print(f"  ⚠️  No data returned for '{keyword}'. Skipping.")
        continue

    if "isPartial" in df_raw.columns:
        df_raw = df_raw.drop(columns=["isPartial"])

    df_raw = df_raw.reset_index()
    df_raw.columns = ["Date", "Search_Volume"]
    df_raw["Keyword"] = keyword

    print(f"  ✓  {len(df_raw)} rows  |  "
          f"max={df_raw['Search_Volume'].max()}  "
          f"mean={df_raw['Search_Volume'].mean():.1f}")

    all_rows.append(df_raw)

    if keyword != KEYWORDS[-1]:
        print("  Pausing 10s before next keyword …")
        time.sleep(10)

# ── CELL 8: Combine and reshape ───────────────────────────────
if not all_rows:
    raise RuntimeError(
        "No data was retrieved.\n"
        "Try: Runtime → Disconnect and delete runtime, reconnect, then re-run."
    )

df_combined = pd.concat(all_rows, ignore_index=True)
df_final = df_combined[["Date", "Keyword", "Search_Volume"]].copy()
df_final["Date"] = pd.to_datetime(df_final["Date"])
df_final = df_final.sort_values(["Date", "Keyword"]).reset_index(drop=True)

# ── CELL 9: Preview ───────────────────────────────────────────
print("\n── Sample output (first 10 rows) ──────────────────────────")
print(df_final.head(10).to_string(index=False))
print(f"\nTotal rows : {len(df_final)}")
print(f"Date range : {df_final['Date'].min().date()} → {df_final['Date'].max().date()}")
print(f"Keywords   : {df_final['Keyword'].unique().tolist()}")

# ── CELL 10: Export ───────────────────────────────────────────
df_final.to_csv(OUTPUT_FILE, index=False)
print(f"\n✅ Saved to '{OUTPUT_FILE}'")

# ── CELL 11: Auto-download in Colab (optional) ────────────────
# from google.colab import files
# files.download(OUTPUT_FILE)

# ── CELL 12: Quick visualisation (optional) ───────────────────
# import matplotlib.pyplot as plt
# fig, ax = plt.subplots(figsize=(12, 5))
# for kw, group in df_final.groupby("Keyword"):
#     ax.plot(group["Date"], group["Search_Volume"], label=kw, linewidth=2)
# ax.set_title("Google Search Interest (US) — Apr~May 2025", fontsize=14)
# ax.set_xlabel("Date")
# ax.set_ylabel("Search Interest (0–100)")
# ax.legend()
# ax.grid(True, alpha=0.3)
# plt.tight_layout()
# plt.show()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 5.0 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
blobfile 3.2.0 requires urllib3>=2, but you have urllib3 1.26.20 which is incompatible.
Fetching data from 2025-04-01 2025-05-31 …
Obtaining Google session cookie …
  ✓ NID cookie obtained (238 chars)
  ⚠️  Could not set NID cookie on pytrends' internal session. May encounter rate-limiting.

→ Fetching: 'Coachella' …
  ✓  61 rows  |  max=100  mean=11.7
  Pausing 10s before next keyword …

→ Fetching: 'Alo Yoga' …
  ✓  61 rows  |  max=100  mean=46.0

── Sample output (first 10 rows) ──────────────────────